mount drive

In [ ]:
!git clone https://github.com/THU-KEG/AgentIF.git
%cd AgentIF
!ls

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

install libraries

In [ ]:
! pip install -q openai
! pip install -q torch
! pip install -q transformers
! pip install -q datasets
! pip install -q langgraph
! pip install -q langchain
! pip install -q langchain-openai


path to the AgentIF data

In [ ]:
AGENTIF_FILE_PATH = "/content/drive/MyDrive/TokenReductionProject/Data/eval_criteria_DOWNSAMPLED.pkl"

set up environment

In [ ]:
import os

# env variables
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")  # add under Colab Secrets
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # restricts visibility of gpus to only the first gpu (gpu at index 0)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


imports

In [ ]:
import warnings

# remove warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
import json
from pathlib import Path
from copy import deepcopy
from tqdm.notebook import tqdm
import time
from typing import TypedDict
import asyncio
import random
import re


set up device

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"working on {device}")

agent code

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
class State(TypedDict, total=False):
    goal: str
    plan: str
    research: str
    draft: str
    final: str
    critique: str

In [ ]:
class TooMany429s(Exception):
    pass

async def abatch_retry(
    llm, prompts, *, config=None,
    safety_ms=250,
    timeout_s=120,
    max_total_wait_s=60.0,   # prevents “stuck all night” on one chunk
):
    start = time.monotonic()

    while True:
        try:
            result = await asyncio.wait_for(
                llm.abatch(prompts, config=config),
                timeout=timeout_s
            )
            return result

        except asyncio.TimeoutError:
            # transient stall; short backoff
            await asyncio.sleep(1.0 + random.uniform(0, 0.2))

        except Exception as e:
            msg = (str(e) or "").lower()
            is_429 = ("429" in msg) or ("rate limit" in msg) or ("ratelimit" in msg) or ("tpm" in msg)
            if not is_429:
                raise

            if time.monotonic() - start > max_total_wait_s:
                raise TooMany429s(f"429 loop > {max_total_wait_s}s for batch size {len(prompts)}") from e

            # server-guided sleep
            delay_s = None
            m = re.search(r"try again in (\d+)\s*ms", msg)
            if m:
                delay_s = int(m.group(1)) / 1000.0
            else:
                m = re.search(r"try again in ([0-9.]+)\s*s", msg)
                if m:
                    delay_s = float(m.group(1))
            if delay_s is None:
                delay_s = 1.0

            delay_s += safety_ms / 1000.0
            delay_s += random.uniform(0, 0.05)

            await asyncio.sleep(delay_s)

In [ ]:
max_concurrency = 6

async def abatch_in_chunks(
    llm, prompts, *,
    chunk_size=16,
    min_chunk_size=1,
    config=None
):
    out = []
    i = 0
    cur_chunk = min_chunk_size  # start conservative; ramp up

    success_streak = 0

    while i < len(prompts):
        size = min(cur_chunk, len(prompts) - i)
        chunk = prompts[i:i+size]

        try:
            out.extend(await abatch_retry(llm, chunk, config=config))
            i += size
            success_streak += 1

            # ramp up slowly after a few successes
            if success_streak >= 3 and cur_chunk < chunk_size:
                cur_chunk = min(chunk_size, cur_chunk * 2)
                success_streak = 0

        except TooMany429s:
            success_streak = 0
            # back off quickly
            cur_chunk = max(min_chunk_size, cur_chunk // 2)
            await asyncio.sleep(1.0)  # small pause to let TPM window breathe

    return out

async def planner_node(states: list[State]) -> list[State]:
    prompts = [
        f"Goal: {s['goal']}\n"
        "Create a numbered plan (3-6 steps) with concrete subtasks."
        for s in states
    ]
    msgs = await abatch_in_chunks(llm, prompts, config={"max_concurrency":max_concurrency}) #reduce if hitting rate limits, increase if not using all resources

    out_states = []
    for s, msg in zip(states, msgs):
        s2 = dict(s)
        s2["plan"] = msg.content
        out_states.append(s2)

    return out_states

async def researcher_node(states: list[State]) -> list[State]:
    prompts = [
        f"Goal: {s['goal']}\nPlan:\n{s.get('plan','')}\n\n"
        "List key facts/constraints/assumptions. Keep it short."
        for s in states
    ]
    msgs = await abatch_in_chunks(llm, prompts, config={"max_concurrency": max_concurrency}) #reduce if hitting rate limits, increase if not using all resources

    out_states = []
    for s, msg in zip(states, msgs):
        s2 = dict(s)
        s2["research"] = msg.content
        out_states.append(s2)

    return out_states

async def builder_node(states: list[State]) -> list[State]:
    prompts = [
        f"Goal: {s['goal']}\nPlan:\n{s.get('plan','')}\nResearch:\n{s.get('research','')}\n\n"
        "Write the deliverable now."
        for s in states
    ]
    msgs = await abatch_in_chunks(llm, prompts, config={"max_concurrency": max_concurrency}) #reduce if hitting rate limits, increase if not using all resources

    out_states = []
    for s, msg in zip(states, msgs):
        s2 = dict(s)
        s2["draft"] = msg.content
        out_states.append(s2)

    return out_states

async def critic_node(states: list[State]) -> list[State]:
    prompts = [
        f"Goal: {s['goal']}\nDraft:\n{s.get('draft','')}\n\n"
        "Critique the draft with bullet points (issues + fixes). "
        for s in states
    ]
    msgs1 = await abatch_in_chunks(llm, prompts, config={"max_concurrency": max_concurrency}) #reduce if hitting rate limits, increase if not using all resources

    prompts = [
        f"Goal: {s['goal']}\nDraft:\n{s.get('draft','')}\n\n"
        f"Based on the critique {msg1.content} output a revised FINAL version."
        for s, msg1 in zip(states, msgs1)
    ]

    msgs2 = await abatch_in_chunks(llm, prompts, config={"max_concurrency": max_concurrency}) #reduce if hitting rate limits, increase if not using all resources

    out_states = []
    for s, msg1, msg2 in zip(states, msgs1, msgs2):
        s2 = dict(s)
        s2["critique"] = msg1.content
        s2["final"] = msg2.content
        out_states.append(s2)

    return out_states

In [ ]:
async def run_agentic_system(prompts: list[str]) -> tuple[list[dict],list[str]]: #returns full output list + final output list

    states = [{"goal": p} for p in prompts]
    states = await planner_node(states)
    states = await researcher_node(states)
    states = await builder_node(states)
    states = await critic_node(states)

    full_outputs = []
    outputs = []
    for s in states:
        full_output = {}

        full_output["plan"] = s.get('plan','')
        full_output["research"] = s.get('research','')
        full_output["draft"] = s.get('draft','')
        full_output["critique"] = s.get('critique','')
        full_output["final"] = s.get('final','')

        full_outputs.append(full_output)

        outputs.append(s.get("final", ""))

    return full_outputs, outputs

load the AgentIF data

In [ ]:
p = Path(AGENTIF_FILE_PATH)

def load_agentif_rows(path: Path):
    if path.suffix == ".pkl":
        obj = pd.read_pickle(path)  # could be DataFrame or list[dict]
        if isinstance(obj, pd.DataFrame):
            return obj.to_dict("records")
        return obj
    elif path.suffix == ".json":
        return json.loads(path.read_text())
    elif path.suffix in [".jsonl", ".jl"]:
        return [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

rows = load_agentif_rows(p)
print("Loaded rows:", len(rows))
print("Example rows:", rows[1])
print("Example keys:", rows[1].keys())
print("Example values:", rows[1].values())

run the agentic system on every instruction

In [ ]:
BATCH_SIZE = 32

out_rows = []
batch = []
for r in tqdm(rows): #runs agentic system on all of the outputs
    rr = deepcopy(r)
    full_input = ""
    for query in rr["input"]:
      full_input += str(query.get("content")) + "\n"
    batch.append([full_input, rr])

    #runs whole thing every [batch size] prompts
    if (len(batch) == BATCH_SIZE):
      batched_prompts = [this_rr[0] for this_rr in batch]
      batched_rr = [this_rr[1] for this_rr in batch]
      full_outputs, outputs = await run_agentic_system(batched_prompts) #use await --> run other things
      for row, full_output, output in zip(batched_rr, full_outputs, outputs):
        row["output"] = {"role": "assistant", "content": output, "total_content":full_output, "raw_total_content":full_output}  #total content is all the agentic yap, content is just the final
      out_rows.extend(batched_rr)
      batch = []

#final partial batch
if (len(batch) > 0):
  batched_prompts = [this_rr[0] for this_rr in batch]
  batched_rr = [this_rr[1] for this_rr in batch]
  full_outputs, outputs = await run_agentic_system(batched_prompts) #use await to run other things simultaneously
  for row, full_output, output in zip(batched_rr, full_outputs, outputs):
    row["output"] = {"role": "assistant", "content": output, "total_content":full_output, "raw_total_content":full_output} #total content is all the agentic yap, content is just the final
  out_rows.extend(batched_rr)
  batch = []

In [ ]:
def convert_numpy(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj

In [ ]:
EXPERIMENT_NUM = "0"
AGENT_OUTPUT_PATH_JSON = Path("/content/drive/MyDrive/TokenReductionProject/Results/AgentOutputs/agent_outputs_for_eval_" + EXPERIMENT_NUM +".json")
AGENT_OUTPUT_PATH_JSON.write_text(json.dumps(out_rows, default=convert_numpy, ensure_ascii=False, indent=2))
print("Wrote", AGENT_OUTPUT_PATH_JSON, "with rows:", len(out_rows))

In [ ]:
#import pandas as pd

AGENT_OUTPUT_PATH_PKL = Path("/content/drive/MyDrive/TokenReductionProject/Results/AgentOutputs/agent_outputs_for_eval_" + EXPERIMENT_NUM +".pkl")

pkl_df = pd.DataFrame(out_rows)
pkl_df.to_pickle(AGENT_OUTPUT_PATH_PKL)

print("Wrote", AGENT_OUTPUT_PATH_PKL, "with rows:", len(out_rows))